In [1]:
import os

In [2]:
%pwd

'd:\\End_to_End_ML_project_for_Ride_Demand_Forecasting_and_Marketplace_Optimization\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\End_to_End_ML_project_for_Ride_Demand_Forecasting_and_Marketplace_Optimization'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainingConfig:
    root_dir: Path
    train_file_path: Path
    validation_file_path: Path
    model_file_path: Path
    metrics_file_path: Path
    model_params: dict
    target_column: str

In [6]:
from src.ride_demand_forecasting_and_marketplace_optimization.constants import *
from src.ride_demand_forecasting_and_marketplace_optimization.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH, schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_training_config(self) -> ModelTrainingConfig:
        config = self.config.model_training

        create_directories([config.root_dir])

        model_training_config = ModelTrainingConfig(
            root_dir=config.root_dir,
            train_file_path=config.train_file_path,
            validation_file_path=config.validation_file_path,
            model_file_path=config.model_file_path,
            metrics_file_path=config.metrics_file_path,
            model_params=dict(self.params.model_params),
            target_column=config.target_column
        )

        return model_training_config    

In [9]:
import pickle as pk

In [11]:
fn_path = "artifacts/data_transformation/feature_names.pkl"

In [12]:
with open(fn_path, "rb") as f:
        feature_names = pk.load(f)

In [13]:
feature_names

array(['num__latitude', 'num__longitude', 'num__completed_rides',
       'num__cancelled_rides', 'num__active_drivers',
       'num__available_drivers', 'num__busy_drivers',
       'num__acceptance_rate', 'num__utilization_rate',
       'num__rider_wait_time', 'num__driver_wait_time',
       'num__surge_multiplier', 'num__avg_fare', 'num__avg_tip',
       'num__revenue', 'num__avg_trip_distance', 'num__avg_trip_duration',
       'num__traffic_index', 'num__temperature', 'num__feels_like',
       'num__humidity', 'num__pressure', 'num__visibility',
       'num__rainfall', 'num__snowfall', 'num__wind_speed',
       'num__holiday_flag', 'num__concert_flag', 'num__sports_event_flag',
       'num__festival_flag', 'num__year', 'num__quarter', 'num__month',
       'num__week', 'num__day', 'num__hour', 'num__dayofweek',
       'num__is_weekend', 'num__fuel_price', 'num__cpi',
       'num__unemployment_rate', 'num__dayofyear', 'num__is_month_start',
       'num__is_month_end', 'num__is_quarter_

In [8]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from src.ride_demand_forecasting_and_marketplace_optimization import logger
from src.ride_demand_forecasting_and_marketplace_optimization.utils.common import save_bin, save_json
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

In [18]:
class ModelTraining:
    """
    Ride Demand Forecasting Model Training Component

    Target:
        ride_requests

    Model:
        XGBoost Regressor

    Metrics:
        RMSE
        MAE
        MAPE
        R2 Score
    """

    def __init__(self, config: ModelTrainingConfig):
        self.config = config

    def load_split_data(self, file_path: str) -> pd.DataFrame:

        if not os.path.exists(file_path):
            raise FileNotFoundError(
                f"Split data file not found: {file_path}"
            )

        logger.info(
            f"Loading split data from: {file_path}"
        )

        file_path = str(file_path)

        if file_path.endswith(".parquet"):
            data = pd.read_parquet(file_path)
            logger.info(
                f"Loaded data shape: {data.shape}"
            )
            return data

        if file_path.endswith(".csv"):
            data = pd.read_csv(file_path)
            logger.info(
                f"Loaded data shape: {data.shape}"
            )
            return data

        raise ValueError(
            f"Unsupported file format: {file_path}"
        )

    def _prepare_features(self,df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:

        target_col = self.config.target_column

        if target_col not in df.columns:
            raise KeyError(
                f"Target column '{target_col}' not found"
            )

        X = df.drop(
            columns=[target_col],
            errors="ignore",
        )

        # Remove target leakage columns
        leakage_columns = [
                "num__completed_rides",
                "num__cancelled_rides",
                "num__revenue",
                "num__supply_demand_ratio",
                "num__marketplace_gap",
                "num__driver_shortage",
                "num__marketplace_pressure",
                "num__surge_x_demand",
                "num__traffic_x_demand",
                "num__borough_avg_demand",
                "num__zone_avg_demand",
                "num__revenue_per_request"
                    ]
        

        leakage_columns = [
            col
            for col in leakage_columns
            if col in X.columns
        ]

        if leakage_columns:
            logger.warning(
                f"Removing leakage columns: "
                f"{leakage_columns}"
            )

            X = X.drop(
                columns=leakage_columns
            )

        # Convert datetime columns
        datetime_cols = X.select_dtypes(
            include=["datetime64[ns]"]
        ).columns.tolist()

        for col in datetime_cols:
            X[col] = pd.to_datetime(
                X[col]
            ).astype("int64") // 10**9

        y = df[target_col]
        
        logger.info(
            f"Prepared features with shape: {X.shape}, "
            f"target shape: {y.shape}"
        )

        return X, y

    def train_model(
        self,
        train_df: pd.DataFrame,
        val_df: pd.DataFrame,
    ) -> XGBRegressor:

        logger.info(
            "Preparing training features"
        )

        X_train, y_train = self._prepare_features(
            train_df
        )

        X_val, y_val = self._prepare_features(
            val_df
        )

        logger.info(
            "Initializing XGBoost Regressor"
        )

        model = XGBRegressor(
            **self.config.model_params
        )

        logger.info(
            "Starting model training"
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[
                (
                    X_val,
                    y_val,
                )
            ],
            verbose=False,
        )

        logger.info(
            "Model training completed"
        )

        return model

    def evaluate_model(
        self,
        model: XGBRegressor,
        df: pd.DataFrame,
    ) -> dict:

        logger.info(
            "Evaluating model"
        )

        X, y = self._prepare_features(df)

        y_pred = model.predict(X)

        rmse = np.sqrt(
            mean_squared_error(
                y,
                y_pred,
            )
        )

        mae = mean_absolute_error(
            y,
            y_pred,
        )

        mape = (
            np.mean(
                np.abs(
                    (y - y_pred)
                    / np.maximum(y, 1)
                )
            )
            * 100
        )

        r2 = r2_score(
            y,
            y_pred,
        )

        metrics = {
            "rmse": float(rmse),
            "mae": float(mae),
            "mape": float(mape),
            "r2_score": float(r2),
        }

        return metrics

    def save_model(
        self,
        model: XGBRegressor,
    ) -> None:

        model_path = Path(
            self.config.model_file_path
        )

        model_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        save_bin(
            data=model,
            path=model_path,
        )

        logger.info(
            f"Saved trained model at: "
            f"{model_path}"
        )

    def save_metrics(
        self,
        metrics: dict,
    ) -> None:

        metrics_path = Path(
            self.config.metrics_file_path
        )

        metrics_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        save_json(
            path=metrics_path,
            data=metrics,
        )

        logger.info(
            f"Saved metrics at: "
            f"{metrics_path}"
        )

    def initiate_model_training(
        self,
    ) -> bool:

        try:

            logger.info(
                "Loading train dataset"
            )

            train_df = self.load_split_data(
                self.config.train_file_path
            )

            logger.info(
                "Loading validation dataset"
            )

            val_df = self.load_split_data(
                self.config.validation_file_path
            )

            model = self.train_model(
                train_df=train_df,
                val_df=val_df,
            )

            self.save_model(model)

            validation_metrics = (
                self.evaluate_model(
                    model=model,
                    df=val_df,
                )
            )

            best_iteration = None

            try:
                booster = model.get_booster()

                if hasattr(booster,"best_iteration"):
                    best_iteration = int(
                        booster.best_iteration
                    )

            except Exception:
                pass

            metrics = {
                "validation": validation_metrics,
                #"test": test_metrics,
                "best_iteration": best_iteration,
            }

            self.save_metrics(metrics)

            logger.info(
                "Model training completed successfully"
            )

            return True

        except Exception as e:

            logger.exception(
                f"Error during model training: {e}"
            )

            raise

In [19]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_training_config()
    model_trainer_config = ModelTraining(config=model_trainer_config)
    model_trainer_config.initiate_model_training()
except Exception as e:
    raise e

[2026-07-17 20:05:15,114: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-17 20:05:15,122: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-17 20:05:15,151: INFO: common: yaml file: config\schema.yaml loaded successfully]
[2026-07-17 20:05:15,156: INFO: common: created directory at: artifacts]
[2026-07-17 20:05:15,158: INFO: common: created directory at: artifacts/model_training]
[2026-07-17 20:05:15,160: INFO: 1223458712: Loading train dataset]
[2026-07-17 20:05:15,162: INFO: 1223458712: Loading split data from: artifacts/data_transformation/splits/train.parquet]
[2026-07-17 20:05:15,230: INFO: 1223458712: Loaded data shape: (122317, 106)]
[2026-07-17 20:05:15,231: INFO: 1223458712: Loading validation dataset]
[2026-07-17 20:05:15,235: INFO: 1223458712: Loading split data from: artifacts/data_transformation/splits/validation.parquet]
[2026-07-17 20:05:15,278: INFO: 1223458712: Loaded data shape: (26211, 106)]
[2026-07-17 20:05:15,280: INFO: 